In [0]:
%sql
-- Spec 4.2 (Nov 12 2026 re-read): DAYS_SINCE_LAST_MEETING_CALENDLY per NORMALIZED
-- INVITEE EMAIL — LOWER(TRIM(INVITEE_EMAIL)) — recency from the latest
-- EVENT_CREATED_AT (booking recency, not event start; no past-only filter: a
-- future booking's creation still counts).
-- GRAIN CHANGE (Nov 12 2026): the Nov 1 build keyed rows per lead_id via the
-- 2_8 identity spine (hash OR email bridge, coaching-filtered) — spec-faithful
-- form is ONE ROW PER invitee email, population = every invitee (no status
-- filter; coaching filtering happens downstream via lead_dim.customer_emails).
-- VERIFIED Nov 12: INVITEE_EMAIL holds NO hashes and NO masked values —
-- 0 NULL / 0 bare 32-hex / 0 masked across 102,597 rows (hashes live in
-- INVITEE_NAME, a separate column this metric does not need — every row
-- carries a real-format email). 50,986 distinct normalized emails; some are
-- Calendly synthetic placeholders (e.g. phone-derived @gmail.com) — kept as
-- recorded; normalization is LOWER/TRIM only. Downstream mapping via lead_dim:
-- 3,834 emails hit coaching-client rows, reaching 1,080 / 1,381 CC leads
-- (vs 1,132 via the old spine hash+email bridge — the 52-lead delta is a
-- lead_dim email-coverage gap, not a 3_10 defect).
-- Dedup for duplicate EVENT_URI per convention: INVITEE_UPDATED_AT, then
-- EVENT_CREATED_AT, then bronze_insert_date (no-op for MAX — event-grain hygiene).
-- NOTE (deferred job cleanup): consumers 3_13/3_14 currently join this table
-- by lead_id — they must re-key to email via lead_dim.customer_emails once
-- this ships.
CREATE OR REPLACE TABLE crm_ingestion.gold.days_since_last_meeting_calendly AS (
WITH calendly_events AS (
    SELECT
        LOWER(TRIM(INVITEE_EMAIL)) AS invitee_email,
        EVENT_CREATED_AT
    FROM
        crm_ingestion.silver.calendly_scheduled_events_processed
    WHERE
        INVITEE_EMAIL IS NOT NULL
        AND EVENT_CREATED_AT IS NOT NULL
    -- Spec dedup for duplicate EVENT_URI: INVITEE_UPDATED_AT, then EVENT_CREATED_AT, then insert date
    QUALIFY ROW_NUMBER() OVER (PARTITION BY EVENT_URI ORDER BY INVITEE_UPDATED_AT DESC, EVENT_CREATED_AT DESC, bronze_insert_date DESC) = 1
)
SELECT
    invitee_email,
    DATE(MAX(EVENT_CREATED_AT)) AS last_booking_date,
    DATEDIFF(CURRENT_DATE, DATE(MAX(EVENT_CREATED_AT))) AS days_since_last_meeting_calendly
FROM
    calendly_events
GROUP BY
    invitee_email
)